In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest

from xgboost import XGBClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

sns.set_theme(style="whitegrid")

In [3]:
ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED_PATH = ROOT / "data" / "processed"
MODEL_PATH = ROOT / "models"

data = joblib.load(
    PROCESSED_PATH / "train_test_data.pkl"
)

smote_data = joblib.load(
    PROCESSED_PATH / "smote_training_data.pkl"
)

X_train = data["X_train"]
X_test = data["X_test"]

y_train = data["y_train"]
y_test = data["y_test"]

X_train_smote = smote_data["X_train_smote"]
y_train_smote = smote_data["y_train_smote"]

print("Original training:", X_train.shape)
print("SMOTE training:", X_train_smote.shape)
print("Testing:", X_test.shape)

print("\nOriginal training distribution:")
print(y_train.value_counts())

print("\nSMOTE distribution:")
print(pd.Series(y_train_smote).value_counts())

Original training: (226980, 30)
SMOTE training: (453204, 30)
Testing: (56746, 30)

Original training distribution:
Class
0    226602
1       378
Name: count, dtype: int64

SMOTE distribution:
Class
0    226602
1    226602
Name: count, dtype: int64


In [4]:
logistic_smote = LogisticRegression(
    max_iter=2000,
    random_state=42
)

logistic_smote.fit(
    X_train_smote,
    y_train_smote
)

print("Logistic Regression with SMOTE trained!")

Logistic Regression with SMOTE trained!


In [5]:
logistic_weighted = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

logistic_weighted.fit(
    X_train,
    y_train
)

print("Weighted Logistic Regression trained!")

Weighted Logistic Regression trained!


In [6]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Genuine transactions:", negative_count)
print("Fraudulent transactions:", positive_count)

print("Scale positive weight:", scale_pos_weight)

Genuine transactions: 226602
Fraudulent transactions: 378
Scale positive weight: 599.4761904761905


In [7]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train
)

print("XGBoost model trained!")

XGBoost model trained!


In [8]:
# Keep only genuine transactions
X_train_genuine = X_train[y_train == 0]

# Estimate expected fraud proportion
fraud_rate = y_train.mean()

isolation_model = IsolationForest(
    n_estimators=200,
    contamination=float(fraud_rate),
    random_state=42,
    n_jobs=-1
)

isolation_model.fit(X_train_genuine)

print("Isolation Forest trained!")
print("Genuine training samples:", len(X_train_genuine))

Isolation Forest trained!
Genuine training samples: 226602


In [9]:
joblib.dump(
    logistic_smote,
    MODEL_PATH / "logistic_smote.pkl"
)

joblib.dump(
    logistic_weighted,
    MODEL_PATH / "logistic_weighted.pkl"
)

joblib.dump(
    xgb_model,
    MODEL_PATH / "xgboost_model.pkl"
)

joblib.dump(
    isolation_model,
    MODEL_PATH / "isolation_forest.pkl"
)

print("All four model configurations saved successfully!")

All four model configurations saved successfully!
